### Data Ingestion

In [1]:
### document Structure 

from langchain_core.documents import Document

In [2]:
## create a simple txt file 
import os
os.makedirs("../data/text_files", exist_ok=True)

In [3]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredMarkdownLoader

loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.md", 
    show_progress=True, 
    loader_cls=UnstructuredMarkdownLoader,
    loader_kwargs={"mode": "elements"},
    )
docs = loader.load()
docs

c:\Users\Testing\.local\bin\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 3/3 [00:30<00:00, 10.21s/it]


[Document(metadata={'source': '..\\data\\text_files\\AI Bootcamp Journey & Learning Path.docx.md', 'languages': ['eng'], 'file_directory': '..\\data\\text_files', 'filename': 'AI Bootcamp Journey & Learning Path.docx.md', 'filetype': 'text/markdown', 'last_modified': '2026-04-13T12:04:55', 'category': 'Image', 'element_id': '2bde646e1e30056fe5b363898ad64326'}, page_content=''),
 Document(metadata={'source': '..\\data\\text_files\\AI Bootcamp Journey & Learning Path.docx.md', 'emphasized_text_contents': ['Bootcamp Journey'], 'emphasized_text_tags': ['b'], 'languages': ['eng'], 'file_directory': '..\\data\\text_files', 'filename': 'AI Bootcamp Journey & Learning Path.docx.md', 'filetype': 'text/markdown', 'last_modified': '2026-04-13T12:04:55', 'category': 'UncategorizedText', 'element_id': '04e90cc25ff09f52942ef1edd3eebee7'}, page_content='Bootcamp Journey'),
 Document(metadata={'source': '..\\data\\text_files\\AI Bootcamp Journey & Learning Path.docx.md', 'languages': ['eng'], 'file_di

### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [4]:
import os
from pathlib import Path
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:


def load_markdown_files(directory: str) -> list:
    """
    Recursively loads all markdown files from a directory,
    and enriches each document with source metadata.
    """
    docs = []
    directory_path = Path(directory)

    # Find all markdown files recursively
    markdown_files = list(directory_path.rglob("*.md"))

    print(f"📂 Found {len(markdown_files)} markdown files.")

    for file_path in markdown_files:
        print(f"\nProcessing {file_path.name}")
        try:
            loader = UnstructuredMarkdownLoader(str(file_path))
            file_docs = loader.load()

            # Add source metadata to each document
            for doc in file_docs:
                doc.metadata['source_file'] = file_path.name
                doc.metadata['file_type'] = 'markdown'
                
            docs.extend(file_docs)
            print(f"✅ Loaded: {len(file_docs)} pages from {file_path.name}")

        except Exception as e:
            print(f"❌ Error loading {file_path.name}: {e}")

    print(f"\n📄 Total documents loaded: {len(docs)}")
    return docs


# --- Usage ---
docs = load_markdown_files("../data")


📂 Found 3 markdown files.

Processing AI Bootcamp Journey & Learning Path.docx.md
✅ Loaded: 1 pages from AI Bootcamp Journey & Learning Path.docx.md

Processing Intern FAQ - AI Bootcamp.md
✅ Loaded: 1 pages from Intern FAQ - AI Bootcamp.md

Processing Training For AI Engineer Interns.md
✅ Loaded: 1 pages from Training For AI Engineer Interns.md

📄 Total documents loaded: 3


In [6]:
### Text splitting get into chunks

def split_documents(docs, chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    Splits documents into smaller chunks using RecursiveCharacterTextSplitter.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", "Q:", "---", " "]  # respect FAQ structure
        )
    
    split_docs= text_splitter.split_documents(docs)
    print (f"Split {len(docs)} documents into {len(split_docs)} chunks.")

    # Show example chunk metadata
    if split_docs:
        print("\nExample chunk metadata:")
        print(f"Content: {split_docs[0].page_content[:100]}...")  # show first 100 chars
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [7]:
chunks = split_documents(docs)
chunks

Split 3 documents into 47 chunks.

Example chunk metadata:
Content: Bootcamp Journey

Use this document as a high-level overview of your journey.

This document will re...
Metadata: {'source': '..\\data\\text_files\\AI Bootcamp Journey & Learning Path.docx.md', 'source_file': 'AI Bootcamp Journey & Learning Path.docx.md', 'file_type': 'markdown'}


[Document(metadata={'source': '..\\data\\text_files\\AI Bootcamp Journey & Learning Path.docx.md', 'source_file': 'AI Bootcamp Journey & Learning Path.docx.md', 'file_type': 'markdown'}, page_content='Bootcamp Journey\n\nUse this document as a high-level overview of your journey.\n\nThis document will reference both these aspects:\n\nTechnical Skills Development\n\nCore ML/AI Concepts\n\nGen AI & Data Engineering\n\nMLOps & Deployment\n\nProject-Based Learning\n\nAgile Scrum Methodology\n\nTeam Collaborations\n\nReal-world Applications\n\nProject Timeline\n\nHere is a high-level timeline of your 11-week journey.\n\nWeek 1 - 11 Agenda for AI PM Bootcamp\n\nWeek 1: Learning and Onboarding Study all the AI knowledge:\n\nTraining for AI Engineers\n\nEngineers’ Training Youtube Playlist\n\nTraining of AI Designers\n\nDesigners’ Training Playlist\n\nEngineers: Working on Job Assistant Agent or Discord RAG FAQ Chatbot\n\nMake sure to join Office hours to discuss your thoughts and issues with 

###  Embedding an VectoStoreDB

In [8]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformers."""
    def __init__(self, model_name: str = "multi-qa-mpnet-base-dot-v1"):
        """
        Initializes the embedding manager.
        Args:
            model_name (str): The name of the SentenceTransformer model to use.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            print(f"✅ Loaded embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ Model loaded successfully: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of texts.
        Args:
            texts (List[str]): The list of texts to embed.
        Returns:
            np.ndarray: The generated embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")

        print(f"🔍 Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"✅ Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize embedding manager and generate embeddings for chunks
embedding_manager = EmbeddingManager()
embedding_manager

✅ Loaded embedding model: multi-qa-mpnet-base-dot-v1


Loading weights: 100%|██████████| 199/199 [00:04<00:00, 47.38it/s]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully: 768


### VectorStore

In [10]:
class VectorStore:
    """Manages document embeddings in a Chromadb vector store."""
    def __init__(self, collection_name: str = "documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initializes the Chromadb client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of document embeddings for RAG",
                    "hnsw:space": "cosine"      # 👈 ADDED
                }
            )
            print(f"✅ Initialized vector store collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")
            raise

    def reset_collection(self):
        """Deletes and recreates the collection. Use when switching embedding models."""
        try:
            self.client.delete_collection(self.collection_name)
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={
                    "description": "Collection of document embeddings for RAG",
                    "hnsw:space": "cosine"      # 👈 ADDED
                }
            )
            print(f"✅ Collection '{self.collection_name}' reset successfully.")
        except Exception as e:
            print(f"❌ Error resetting collection: {e}")
            raise

    def add_documents(self, docs: List[Document], embeddings: np.ndarray):
        """Adds documents and their corresponding embeddings to the store."""
        if len(docs) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(docs, embeddings)):
            docs_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(docs_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"✅ Added {len(docs)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"❌ Error adding documents to vector store: {e}")
            raise

In [11]:
vectorstore = VectorStore()
vectorstore.reset_collection()  # clears old collection, recreates with cosine metric

texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks, embeddings)

✅ Initialized vector store collection: documents
Existing documents in collection: 47
✅ Collection 'documents' reset successfully.
🔍 Generating embeddings for 47 texts...


Batches: 100%|██████████| 2/2 [03:34<00:00, 107.41s/it]


✅ Generated embeddings with shape: (47, 768)
✅ Added 47 documents to the vector store.
Total documents in collection after addition: 47


### Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """Retrieves relevant documents from the vector store based on a query."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initializes the RAG retriever.
        Args:
            vector_store (VectorStore): The vector store instance to retrieve from.
            embedding_manager (EmbeddingManager): The embedding manager to generate query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieves relevant documents based on the query.
        
        Args:
            query (str): The input query to search for.
            top_k (int): The number of top relevant documents to return.
            score_threshold (float): The minimum similarity score for a document to be considered relevant.
        Returns:
            List[Dict[str, Any]]: A list of retrieved documents relevant to the query.
        """
        print(f"🔍 Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]  # Get the first (and only) embedding

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            # Process results
            retrieved_docs = []

            if results ['documents'] and results['documents'][0]:  # Check if there are any documents returned
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, doc_text, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distance to similarity score (assuming distance is between 0 and 1)
                    similarity_score = 1 - distance  # Convert distance to similarity

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "text": doc_text,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1  # Add rank based on order returned
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("⚠️ No documents retrieved from vector store.")
            return retrieved_docs
        except Exception as e:
            print(f"❌ Error retrieving documents: {e}")
            return []
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [13]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)
results = rag_retriever.retrieve("What are the responsibilities of the lead engineer?")
results

🔍 Retrieving documents for query: 'What are the responsibilities of the lead engineer?' with top_k=5 and score_threshold=0.0
🔍 Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:02<00:00,  2.61s/it]


✅ Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)


[{'id': 'doc_aeb65602_2',
  'text': 'Make sure to join Office hours to discuss your thoughts and issues with these projects.\n\nWeek 3: Continue development\n\nDesigner building High-Fidelity Designs\n\nUser Interviews (continue)\n\nPrepare for Week 4 presentation to Engineers/Data Scientists\n\nEngineers: Working on Job Assistant Agent or Discord RAG FAQ Chatbot\n\nMake sure to join Office hours to discuss your thoughts and issues with these projects.\n\nSubmit your code and video walk through of code logic and provide a running example by filling out this Google Form\n\nWeek 4: Prepare and Join Pitch Day & Ranking\n\nDesigners & PM present to Engineers\n\nEngineers join Zoom Pitch Day and fill out Google Ranking form. (Lead Engineers get Ranking choice priority)\n\nLead Engineer Responsibilities \\=\n\nLead System Architecture Design (Bring thoughts to Office Hours)\n\nDetermine version control process in shared repository (Github) [And Git locally]\n\nAssist other Engineers (not cod

### Integration Vectordb Context Pipeline With LLM output 

In [16]:
### Discord RAG Assistant with Groq LLM
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
import os
from dotenv import load_dotenv

load_dotenv()

# ── LLM Setup ────────────────────────────────────────────────────────────────
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=512  # Keep responses concise for Discord
)

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are Aria, a friendly and knowledgeable Discord assistant for the AI PM Bootcamp. 
You help students, interns, and team members get quick answers about the bootcamp.

PERSONALITY:
- Warm, encouraging and professional
- Speak naturally like a helpful team member, not a robot
- Use light formatting (bold, bullet points) to make answers readable in Discord

RULES:
1. Always answer in a conversational, assistant-like tone.
2. Use the context provided to give the most complete answer you can.
3. If the context partially answers the question — answer what you can, then say:
   "For more details, feel free to ask or reach out to a human assistant! 🙋"
4. If the context has absolutely no relevance to the question, say:
   "Hmm, I don't have information on that just yet. Please reach out to a human assistant for help 🙋"
5. Never mention documents, scores, rankings, or that you're searching anything.
6. Never say "based on the context provided" — just answer naturally.
7. Keep answers concise but complete — no walls of text.

ANSWER FORMAT (for Discord):
- Use **bold** for key terms
- Use bullet points for lists
- Keep responses under 10 lines where possible
- Add a relevant emoji occasionally to keep it friendly 😊
"""

# ── Fallback Message ──────────────────────────────────────────────────────────
FALLBACK_NO_DOCS    = "⚠️ I couldn't find any relevant information. Please contact a human assistant for help. 🙋"
FALLBACK_LLM_ERROR  = "⚠️ Something went wrong on my end. Please try again or contact a human assistant. 🙋"

# ── RAG Function ──────────────────────────────────────────────────────────────
def rag_answer(query: str, retriever, top_k: int = 5, score_threshold: float = 0.2, llm=llm) -> str:
    """
    Retrieves relevant documents and generates a grounded answer.
    Falls back gracefully if no context is found or LLM fails.

    Args:
        query          : The user's question.
        retriever      : Your vector store retriever instance.
        top_k          : Number of documents to retrieve.
        score_threshold: Minimum similarity score to accept a document.

    Returns:
        str: A concise, Discord-friendly answer.
    """

    # Step 1 — Retrieve
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)

    if not retrieved_docs:
        return FALLBACK_NO_DOCS

    # Step 2 — Build context (clean, no rank/score noise for the LLM)
    context = "\n\n".join([doc["text"] for doc in retrieved_docs])

    # Step 3 — Build messages
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {query}")
    ]

    # Step 4 — Generate
    try:
        response = llm.invoke(messages)
        answer = response.content.strip()

        # Safety net: if LLM returns empty string
        if not answer:
            return FALLBACK_NO_DOCS

        return answer

    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return FALLBACK_LLM_ERROR

In [18]:
answer=rag_answer("what is the team size for the AI PM Bootcamp?", rag_retriever)
print(answer)

🔍 Retrieving documents for query: 'what is the team size for the AI PM Bootcamp?' with top_k=5 and score_threshold=0.2
🔍 Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


✅ Generated embeddings with shape: (1, 768)
Retrieved 5 documents (after filtering)
**Team Size & Composition** 🤝

Typically, a team in the AI PM Bootcamp consists of 8-10 members. However, larger-scope projects may be assigned bigger teams.
